# ATLC Phase 14 - Runtime Security Simulation

This notebook documents and tests the Phase 14 security layer for the ATLC project.

The goal is not only to run source code, but also to explain the security principle before each implementation step. The notebook follows the required **Text + Code** structure:

1. Explain the security problem in the ATLC runtime pipeline.
2. Define independent ATLC-style test data.
3. Authenticate PLAN messages using HMAC-SHA256.
4. Reject tampered, forged, replayed, and stale PLAN messages.
5. Verify tamper-evident audit logs using a SHA-256 hash chain.
6. Load runtime evidence generated by the ATLC pipeline.
7. Summarize execution results for the final report.


## 1. Security Motivation

The ATLC system processes traffic video, detects vehicles, estimates density, and generates traffic-light control decisions. In a real deployment, these decisions may be transmitted to a controller, stored in logs, or displayed on a monitoring dashboard.

This creates several security risks:

- **Message tampering:** an attacker modifies a PLAN message after it is generated.
- **Command forgery:** an attacker creates a fake PLAN message.
- **Replay attack:** an attacker records an old valid PLAN and sends it again.
- **Timestamp abuse:** an attacker uses a stale message outside the expected time window.
- **Audit-log tampering:** an attacker modifies logs after a run to hide evidence.

Phase 14 adds a software-level security experiment to test these risks before hardware integration.


## 2. Notebook Setup

This notebook assumes it is executed from the ATLC project root. If it is opened from `crypto_research/notebooks`, the next cell automatically moves to the repository root by searching for the `pc_app` and `experiments` folders.


In [ ]:
from pathlib import Path
import os
import sys
import json
import time
import subprocess


def find_project_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "pc_app").exists() and (candidate / "experiments").exists():
            return candidate
    raise RuntimeError("Cannot find ATLC project root. Please run this notebook inside the repo.")

PROJECT_ROOT = find_project_root(Path.cwd())
os.chdir(PROJECT_ROOT)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)
print("Python:", sys.executable)


## 3. Independent ATLC Test Data

The PLAN message below is not a shared public cryptography sample. It is an ATLC-style control decision containing green times, yellow time, all-red time, timestamp, nonce, and a monotonically increasing `plan_id`.

This data represents the type of message that a runtime scheduler or traffic-light controller would need to authenticate before execution.


In [ ]:
from pc_app.security.secure_runtime import (
    create_secure_plan,
    verify_secure_plan,
)

SECRET_KEY = b"atlc_phase14_demo_secret_key_not_for_production"
current_timestamp = int(time.time())

base_plan = {
    "message_type": "PLAN",
    "plan_id": 1,
    "green_a": 40,
    "green_b": 10,
    "yellow": 3,
    "all_red": 1,
    "timestamp": current_timestamp,
    "nonce": "phase14-demo-nonce-001",
}

secure_plan = create_secure_plan(
    plan=base_plan,
    secret_key=SECRET_KEY,
)

print(json.dumps(secure_plan, indent=2))


## 4. Valid PLAN Verification

A valid PLAN should be accepted when:

- the HMAC is correct,
- the `plan_id` is newer than the last accepted ID,
- the nonce has not been used before,
- the timestamp is inside the allowed time window.


In [ ]:
valid_ok, valid_reason = verify_secure_plan(
    plan=secure_plan,
    secret_key=SECRET_KEY,
    last_seen_plan_id=0,
    used_nonces=set(),
    allowed_time_window_seconds=300,
)

print("Valid PLAN accepted:", valid_ok)
print("Reason:", valid_reason)
assert valid_ok is True
assert valid_reason == "OK"


## 5. Attack Simulation Cases

The next cell executes the official Phase 14 attack simulator. It tests:

1. Valid PLAN acceptance.
2. Tampered PLAN rejection.
3. Replay attack rejection.
4. Forged MAC rejection.
5. Timestamp tamper rejection.
6. Original audit log verification.
7. Tampered audit log rejection.

The simulator also writes report-ready evidence files under:

`outputs/security/phase14_runtime_security/`


In [ ]:
from experiments.attack_simulator import run_attack_simulation

summary = run_attack_simulation()
print(json.dumps(summary, indent=2))

assert summary["valid_plan_accepted"] is True
assert summary["tampered_plan_rejected"] is True
assert summary["replay_rejected"] is True
assert summary["forged_mac_rejected"] is True
assert summary["timestamp_tamper_rejected"] is True
assert summary["audit_log_valid"] is True
assert summary["tampered_audit_log_rejected"] is True


## 6. Test Script Validation

This cell runs the plain Python test script used for Phase 14 validation. It is intentionally designed to work without `pytest`, so the result can be copied directly into the report or presentation.


In [ ]:
result = subprocess.run(
    [sys.executable, "-m", "experiments.test_secure_runtime"],
    cwd=PROJECT_ROOT,
    text=True,
    capture_output=True,
)

print(result.stdout)
if result.stderr:
    print("STDERR:")
    print(result.stderr)

assert result.returncode == 0


## 7. Audit Log Verification

The hash-chained audit log is used to make log tampering visible. Every line contains a `previous_hash` and `current_hash`. If a hacker edits one line after execution, the recomputed hash no longer matches the stored hash.

This section verifies both:

- the original secure audit log,
- the hacker-modified log.


In [ ]:
SECURITY_DIR = PROJECT_ROOT / "outputs" / "security" / "phase14_runtime_security"
secure_log = SECURITY_DIR / "secure_runtime_audit_log.jsonl"
hacker_log = SECURITY_DIR / "hacker_modified_log.jsonl"
tampered_log = SECURITY_DIR / "tampered_secure_runtime_audit_log.jsonl"

print("Security directory:", SECURITY_DIR)
print("Secure log exists:", secure_log.exists())
print("Hacker-modified log exists:", hacker_log.exists())
print("Auto-tampered log exists:", tampered_log.exists())


In [ ]:
def run_verify(log_path: Path) -> str:
    result = subprocess.run(
        [sys.executable, "-m", "experiments.verify_audit_log", "--log", str(log_path)],
        cwd=PROJECT_ROOT,
        text=True,
        capture_output=True,
    )
    output = result.stdout + result.stderr
    print(output)
    assert result.returncode in (0, 1)
    return output

print("=== Original secure audit log ===")
secure_output = run_verify(secure_log)
assert "Audit log valid: True" in secure_output

print("=== Hacker modified audit log ===")
if hacker_log.exists():
    hacker_output = run_verify(hacker_log)
    assert "Audit log valid: False" in hacker_output
else:
    print("hacker_modified_log.jsonl has not been created yet. Copy the secure log and edit one payload field manually.")


## 8. Runtime Evidence from ATLC

Phase 14 is not only a cryptography script. It is connected to the ATLC runtime pipeline through generated runtime logs, plots, and output video evidence.

The expected Phase 14 runtime paths are:

- `outputs/pipeline_runs/yolo26n_custom_phase14_security_runtime/logs/traffic_runtime_log.csv`
- `outputs/figures/yolo26n_custom_phase14_security_runtime/`
- `outputs/reports/yolo26n_custom_phase14_security_runtime_summary.md`

If these files do not exist yet, run the ATLC main pipeline and runtime plotting script first.


In [ ]:
import pandas as pd

runtime_csv = PROJECT_ROOT / "outputs" / "pipeline_runs" / "yolo26n_custom_phase14_security_runtime" / "logs" / "traffic_runtime_log.csv"
figures_dir = PROJECT_ROOT / "outputs" / "figures" / "yolo26n_custom_phase14_security_runtime"
summary_md = PROJECT_ROOT / "outputs" / "reports" / "yolo26n_custom_phase14_security_runtime_summary.md"

print("Runtime CSV:", runtime_csv, runtime_csv.exists())
print("Figures dir:", figures_dir, figures_dir.exists())
print("Summary:", summary_md, summary_md.exists())

if runtime_csv.exists():
    df = pd.read_csv(runtime_csv)
    print("Runtime log shape:", df.shape)
    display(df.head())
else:
    print("Runtime CSV not found. Generate it with pc_app.main using the Phase 14 .env paths.")


## 9. Runtime Metrics Summary

This section summarizes key runtime metrics if the ATLC runtime CSV exists. These values can be reused in the final report as evidence that the security experiment is connected to a real runtime pipeline.


In [ ]:
if runtime_csv.exists():
    metrics = {
        "samples": int(len(df)),
        "avg_processing_fps": float(df["processing_fps"].mean()) if "processing_fps" in df.columns else None,
        "avg_direction_a_count": float(df["direction_a_count"].mean()) if "direction_a_count" in df.columns else None,
        "avg_direction_b_count": float(df["direction_b_count"].mean()) if "direction_b_count" in df.columns else None,
        "max_active_green_a": int(df["active_green_a"].max()) if "active_green_a" in df.columns else None,
        "max_active_green_b": int(df["active_green_b"].max()) if "active_green_b" in df.columns else None,
    }
    print(json.dumps(metrics, indent=2))
else:
    metrics = {}
    print("No runtime metrics available yet.")


## 10. Report Figures

The final report can include images from `docs/security/figures` and runtime plots from `outputs/figures/yolo26n_custom_phase14_security_runtime`.

Recommended report evidence:

- Runtime terminal screenshot.
- Output video frame with detection boxes and signal overlay.
- Runtime FPS plot.
- Direction count plot.
- Green time plot.
- Runtime phase timeline.
- Attack simulation summary screenshot.
- Valid audit log verification screenshot.
- Hacker-modified audit log rejection screenshot.


In [ ]:
from IPython.display import Image, display

candidate_figures = [
    PROJECT_ROOT / "docs" / "security" / "figures" / "fig_phase14_runtime_annotated_video_frame.png",
    PROJECT_ROOT / "docs" / "security" / "figures" / "fig_phase14_attack_simulation_summary.png",
    PROJECT_ROOT / "docs" / "security" / "figures" / "fig_phase14_hacker_modified_log_rejected.png",
    figures_dir / "runtime_fps.png",
    figures_dir / "direction_counts.png",
    figures_dir / "green_times.png",
    figures_dir / "runtime_phase_timeline.png",
]

for fig in candidate_figures:
    print(fig, "exists=", fig.exists())
    if fig.exists():
        display(Image(filename=str(fig)))


## 11. Security Interpretation

The test results show the following:

- A valid PLAN is accepted, so the security layer does not block legitimate runtime commands.
- A tampered PLAN is rejected with `INVALID_MAC`, proving that HMAC-SHA256 detects message modification.
- A replayed PLAN is rejected with `REPLAY_OR_OLD_PLAN_ID`, proving that plan ordering is enforced.
- A forged MAC is rejected, proving that an attacker cannot create a valid command without the secret key.
- A stale timestamp is rejected with `TIMESTAMP_OUT_OF_WINDOW`.
- A modified audit log is rejected with a hash mismatch, proving that the log is tamper-evident.

This confirms that Phase 14 satisfies the key software-level security goals: authentication, integrity, replay protection, and tamper-evident logging.


## 12. Limitations

This notebook validates the Phase 14 security layer at software simulation level. The current limitations are:

- The security layer is not yet fully integrated into the real UART transmission path.
- Hardware traffic-light testing is not included in this notebook.
- The HMAC key is a demo key and not managed by a production key-management system.
- AES-GCM and RSA-OAEP are demonstrated in the crypto research layer, but the runtime PLAN simulator focuses mainly on HMAC, replay protection, and audit logs.
- The web dashboard authentication and MongoDB account storage are separate extensions and should be documented by the teammate responsible for the web module.


## 13. Conclusion

This notebook demonstrates that the ATLC project has a testable runtime security layer. The implementation is connected to project-specific ATLC data structures, runtime logs, and security evidence files.

The Phase 14 contribution is therefore not only theoretical. It includes executable attack simulations, audit-log verification, generated evidence, and report-ready results.
